In [1]:
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG
# =========================
DATA_PATH = "/kaggle/input/news-article-v4/development.csv"
RANDOM_STATE = 42
USE_ONLY_FIRST_FOLD = True

RULE_CONFIGS = [
    {"min_support": 20, "purity": 0.90, "priority": "best_purity_then_freq"},
    {"min_support": 30, "purity": 0.90, "priority": "best_purity_then_freq"},
    {"min_support": 40, "purity": 0.90, "priority": "best_purity_then_freq"},
    {"min_support": 30, "purity": 0.92, "priority": "best_purity_then_freq"},
    {"min_support": 30, "purity": 0.95, "priority": "best_purity_then_freq"},
    {"min_support": 30, "purity": 0.90, "priority": "freq_then_purity"},
]

# =========================
# LOAD + TIMESTAMP DROP
# =========================
df = pd.read_csv(DATA_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df[df["timestamp"].notna()].reset_index(drop=True)

print("Samples after timestamp drop:", len(df))

# =========================
# BASIC FIXES
# =========================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)

# =========================
# TEXT
# =========================
df["text"] = (df["title"] + " " + df["article"]).str.lower()

# =========================
# NUMERIC FEATURES
# =========================
df["n_tokens"] = df["article"].str.split().str.len()
df["title_len"] = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

df["year"]  = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["dow"]   = df["timestamp"].dt.dayofweek

# =========================
# X / y
# =========================
X = df[
    ["source", "text",
     "n_tokens", "title_len", "article_len",
     "title_ratio", "year", "month", "dow"]
]
y = df["label"].astype(int)

# =========================
# TOKENIZER (RULES)
# =========================
def tokenize_for_rules(text):
    return text.split()

# =========================
# RULE MINING
# =========================
def mine_pure_rules(texts, labels, min_support, purity_thr):
    counts = defaultdict(lambda: Counter())

    for txt, y in zip(texts, labels):
        for tok in set(tokenize_for_rules(txt)):
            counts[tok][y] += 1

    rule_token_to_class = {}
    rule_meta = {}

    for tok, c in counts.items():
        total = sum(c.values())
        if total < min_support:
            continue

        best_class, best_freq = c.most_common(1)[0]
        purity = best_freq / total

        if purity >= purity_thr:
            rule_token_to_class[tok] = best_class
            rule_meta[tok] = (purity, total)

    return rule_token_to_class, rule_meta

# =========================
# APPLY RULES
# =========================
def apply_rules(texts, rule_token_to_class, rule_meta, priority):
    rule_pred = np.full(len(texts), -1, dtype=int)
    matched_token = [None] * len(texts)

    for i, txt in enumerate(texts):
        toks = set(tokenize_for_rules(txt))
        hits = [t for t in toks if t in rule_token_to_class]
        if not hits:
            continue

        if priority == "best_purity_then_freq":
            hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
        else:
            hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

        best = hits[0]
        rule_pred[i] = int(rule_token_to_class[best])
        matched_token[i] = best

    return rule_pred, matched_token

# =========================
# MODEL
# =========================
def make_model():
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1,2),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),
            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3,5),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),
            ("num", StandardScaler(), [
                "n_tokens", "title_len", "article_len",
                "title_ratio", "year", "month", "dow"
            ])
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=2.0,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])

# =========================
# RUN (1 FOLD)
# =========================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for cfg in RULE_CONFIGS:

    print("\n" + "="*80)
    print(f"RULE CONFIG → support={cfg['min_support']} | purity={cfg['purity']} | priority={cfg['priority']}")
    print("="*80)

    for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):

        X_tr, y_tr = X.iloc[tr], y.iloc[tr]
        X_te, y_te = X.iloc[te], y.iloc[te]

        rule_token_to_class, rule_meta = mine_pure_rules(
            X_tr["text"], y_tr,
            cfg["min_support"], cfg["purity"]
        )

        print(f"[Fold {fold_id}] mined rules: {len(rule_token_to_class)}")

        model = make_model()
        model.fit(X_tr, y_tr)

        base_pred = model.predict(X_te)

        rule_pred, matched_token = apply_rules(
            X_te["text"], rule_token_to_class, rule_meta, cfg["priority"]
        )

        final_pred = base_pred.copy()
        mask = rule_pred != -1
        final_pred[mask] = rule_pred[mask]

        print("Macro F1:", f1_score(y_te, final_pred, average="macro"))
        print("Rule coverage:", f"{mask.mean():.3f}", f"({mask.sum()}/{len(mask)})")

        if mask.any():
            print("Rule-only Macro F1:",
                f1_score(y_te[mask], final_pred[mask], average="macro"))

        print("\nPer-class recall:")
        rep = classification_report(y_te, final_pred, digits=3, output_dict=True)
        for c in range(7):
            print(f"  class {c}: recall={rep[str(c)]['recall']:.3f}")

        counter = Counter([t for t in matched_token if t is not None])
        print("Top rule tokens:", counter.most_common(15))

        if USE_ONLY_FIRST_FOLD:
            break


Samples after timestamp drop: 52247

RULE CONFIG → support=20 | purity=0.9 | priority=best_purity_then_freq
[Fold 1] mined rules: 206
Macro F1: 0.7457302194352916
Rule coverage: 0.134 (1400/10450)
Rule-only Macro F1: 0.8565910360698178

Per-class recall:
  class 0: recall=0.711
  class 1: recall=0.808
  class 2: recall=0.847
  class 3: recall=0.655
  class 4: recall=0.918
  class 5: recall=0.633
  class 6: recall=0.761
Top rule tokens: [('/><img', 116), ('details.\\', 64), ('alt="democratic', 38), ('alt="republican', 32), ('afp', 31), ('(hollywood', 30), ('healthday', 29), ('sep.', 27), ('health)', 25), ('yankees', 23), ('inning', 22), ('jun.', 21), ('rodham', 18), ('(pc', 17), ('vista', 16)]

RULE CONFIG → support=30 | purity=0.9 | priority=best_purity_then_freq
[Fold 1] mined rules: 115
Macro F1: 0.7476022360913518
Rule coverage: 0.106 (1110/10450)
Rule-only Macro F1: 0.7830210771312135

Per-class recall:
  class 0: recall=0.712
  class 1: recall=0.811
  class 2: recall=0.848
  class